In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

In [13]:
# Configuration
BASE_URL = "https://www.sports-reference.com"
START_URL = f"{BASE_URL}/cbb/seasons/men/2026-school-stats.html"
OUTPUT_FILE = "player_stats_2026.csv"

def get_team_links():
    print("Fetching team list...")
    response = requests.get(START_URL)
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Find the main school stats table
    table = soup.find("table", {"id": "basic_school_stats"})
    links = []
    
    for row in table.find("tbody").find_all("tr"):
        # Skip header rows that repeat in the middle of the table
        if row.get("class") and "thead" in row.get("class"):
            continue
            
        school_cell = row.find("td", {"data-stat": "school_name"})
        if school_cell and school_cell.find("a"):
            links.append({
                "school": school_cell.text,
                "url": BASE_URL + school_cell.find("a")["href"]
            })
    return links

def scrape_players():
    all_teams_data = []
    teams = get_team_links()
    
    print(f"Found {len(teams)} teams. starting crawl (with 3-second delays to avoid ban)...")
    
    for i, team in enumerate(teams):
        try:
            # Sports-Reference rate limit is 20 requests/min. 
            # 3 seconds between requests = 20 requests/min exactly.
            time.sleep(3) 
            
            # Read all tables on the team page
            # [0] is usually the Roster/Stats table
            tables = pd.read_html(team['url'])
            player_df = tables[5]
            
            # Add school metadata
            player_df["School"] = team['school']
            all_teams_data.append(player_df)
            
            print(f"[{i+1}/{len(teams)}] Scraped {team['school']}")
            
        except Exception as e:
            print(f"Could not scrape {team['school']}: {e}")
            
    # Combine everything
    final_df = pd.concat(all_teams_data, ignore_index=True)
    final_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Success! Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    scrape_players()

Fetching team list...
Found 365 teams. starting crawl (with 3-second delays to avoid ban)...
[1/365] Scraped Abilene Christian
[2/365] Scraped Air Force
[3/365] Scraped Akron
[4/365] Scraped Alabama
[5/365] Scraped Alabama A&M
[6/365] Scraped Alabama State
[7/365] Scraped Albany (NY)
[8/365] Scraped Alcorn State
[9/365] Scraped American
[10/365] Scraped Appalachian State
[11/365] Scraped Arizona
[12/365] Scraped Arizona State
[13/365] Scraped Arkansas
[14/365] Scraped Arkansas State
[15/365] Scraped Arkansas-Pine Bluff
[16/365] Scraped Army
[17/365] Scraped Auburn
[18/365] Scraped Austin Peay
[19/365] Scraped Ball State
[20/365] Scraped Baylor
[21/365] Scraped Bellarmine
[22/365] Scraped Belmont
[23/365] Scraped Bethune-Cookman
[24/365] Scraped Binghamton
[25/365] Scraped Boise State
[26/365] Scraped Boston College
[27/365] Scraped Boston University
[28/365] Scraped Bowling Green
[29/365] Scraped Bradley
[30/365] Scraped Brigham Young
[31/365] Scraped Brown
[32/365] Scraped Bryant
[33/

In [17]:
df_bio = pd.read_csv('player_bio_2026.csv')
df_bio.head()

,Player,#,Class,Pos,Height,Weight,Hometown,High School,RSCI Top 100,Summary,School
0,Bradyn Hubbard,15,SR,F,6-7,225.0,"Tulsa, OK",Tulsa Memorial (OK),NaN,"16.1 Pts, 4.8 Reb, 1.8 Ast",Abilene Christian
1,Rich Smith,4,SR,G,6-4,175.0,"Bronx, NY",Monsignor Scanlan (NY),NaN,"9.6 Pts, 3.9 Reb, 4.7 Ast",Abilene Christian
2,Chilaydrien Newton,5,SO,G,6-3,175.0,"Simsboro, LA",Simsboro (LA); Wasatch Academy (UT),NaN,"9.4 Pts, 2.2 Reb, 0.9 Ast",Abilene Christian
3,Zy Wright,0,SR,G,6-5,NaN,"Lincolnton, GA",Grovetown (GA); Aquinas (GA),NaN,"6.2 Pts, 2.4 Reb, 0.7 Ast",Abilene Christian
4,Yaniel Rivera,2,JR,G,6-4,175.0,"Bayamon, Puerto Rico",Dade Christian (FL),NaN,"7.0 Pts, 1.5 Reb, 2.5 Ast",Abilene Christian


In [18]:
df_bio[df_bio["School"] == "Illinois"]

,Player,#,Class,Pos,Height,Weight,Hometown,High School,RSCI Top 100,Summary,School
1838,Keaton Wagler,23,FR,G,6-6,185.0,"Shawnee, KS",Shawnee Mission Northwest (KS),NaN,"18.1 Pts, 4.9 Reb, 4.3 Ast",Illinois
1839,David Mirkovic,0,FR,F,6-9,250.0,"Nikšić, Montenegro",KK Studentski centar (Montenegro),NaN,"13.1 Pts, 7.8 Reb, 2.6 Ast",Illinois
1840,Andrej Stojakovic,2,JR,G,6-7,190.0,"Carmichael, CA",Carmichael Jesuit (CA),23 (2023),"14.0 Pts, 4.5 Reb, 1.1 Ast",Illinois
1841,Kylan Boswell,4,SR,G,6-2,195.0,"Champaign, IL",Centennial (CA); AZ Compass Prep (AZ),NaN,"13.9 Pts, 4.4 Reb, 3.6 Ast",Illinois
1842,Tomislav Ivisic,13,SO,C,7-1,255.0,"Vodice, Croatia",SC Derby (Montenegro),NaN,"10.5 Pts, 5.7 Reb, 1.6 Ast",Illinois
1843,Zvonimir Ivisic,44,JR,F,7-2,234.0,"Vodice, Croatia",SC Derby (Montenegro),NaN,"6.9 Pts, 5.1 Reb, 0.3 Ast",Illinois
1844,Ben Humrichous,3,SR,F,6-9,220.0,"Tipton, IN",Tipton (IN),NaN,"6.0 Pts, 4.2 Reb, 0.8 Ast",Illinois
1845,Jake Davis,15,JR,F,6-6,210.0,"McCordsville, IN","Cathedral (Indianapolis, IN)",NaN,"5.7 Pts, 2.1 Reb, 0.8 Ast",Illinois
1846,Mihailo Petrovic,77,SO,G,6-3,180.0,"Prokuplje, Serbia",KK Partizan (Serbia); KK Dunav (Serbia); KK Bo...,NaN,"2.1 Pts, 0.7 Reb, 1.3 Ast",Illinois
1847,Brandon Lee,1,FR,G,6-4,195.0,"San Juan, Puerto Rico",The Patrick School (NJ),NaN,"1.9 Pts, 0.4 Reb, 0.2 Ast",Illinois


In [21]:
df_stats = pd.read_csv('player_stats_2026.csv')
df_stats

,Rk,Player,Pos,G,GS,MP,FG,FGA,FG%,3P,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,School
0,1.0,Bradyn Hubbard,F,26,23,27.9,5.3,13.0,0.409,1.4,...,3.7,4.8,1.8,1.9,0.2,2.3,2.8,16.1,NaN,Abilene Christian
1,2.0,Rich Smith,G,25,22,29.9,3.4,7.0,0.483,0.1,...,2.2,3.9,4.7,1.8,0.4,2.8,2.8,9.6,NaN,Abilene Christian
2,3.0,Chilaydrien Newton,G,23,17,24.5,3.4,7.7,0.449,1.1,...,1.5,2.2,0.9,1.2,0.1,1.3,2.3,9.4,NaN,Abilene Christian
3,4.0,Yaniel Rivera,G,22,15,24.3,2.2,6.3,0.345,1.5,...,1.3,1.5,2.5,1.2,0.2,1.5,1.1,7.0,NaN,Abilene Christian
4,5.0,Zy Wright,G,26,4,16.8,2.1,4.8,0.435,0.7,...,1.4,2.4,0.7,0.5,0.2,0.9,1.4,6.2,NaN,Abilene Christian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5367,10.0,Andrew King,G,28,1,18.0,0.8,1.8,0.469,0.2,...,2.4,3.1,3.0,0.5,0.0,1.3,1.4,2.9,NaN,Youngstown State
5368,11.0,Tyler Robinett,F,12,4,11.5,1.2,2.8,0.412,0.2,...,1.2,2.5,0.4,0.2,0.5,0.4,1.7,2.8,NaN,Youngstown State
5369,12.0,Derrick Anderson,G,6,0,5.8,1.2,2.2,0.538,0.2,...,0.5,0.7,1.0,0.8,0.2,0.0,0.2,2.7,NaN,Youngstown State
5370,13.0,Shaheed Solebo,G,7,0,6.4,0.6,1.6,0.364,0.1,...,0.9,1.3,0.9,0.0,0.0,0.7,0.6,2.3,NaN,Youngstown State


In [20]:
df_stats[df_stats["School"] == "Illinois"]

,Rk,Player,Pos,G,GS,MP,FG,FGA,FG%,3P,...,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Awards,School
1746,1.0,Keaton Wagler,G,27,27,33.0,5.3,11.6,0.462,2.5,...,3.0,4.9,4.3,0.9,0.3,1.7,1.9,18.1,NaN,Illinois
1747,2.0,Andrej Stojakovic,G,24,21,27.3,5.3,10.5,0.502,0.7,...,3.2,4.5,1.1,0.4,0.5,1.4,1.3,14.0,NaN,Illinois
1748,3.0,Kylan Boswell,G,20,20,31.6,4.7,10.1,0.465,1.3,...,2.8,4.4,3.6,0.8,0.0,1.3,2.2,13.9,NaN,Illinois
1749,4.0,David Mirkovic,F,27,26,28.8,4.4,9.3,0.476,1.6,...,5.4,7.8,2.6,0.4,0.3,1.9,1.9,13.1,NaN,Illinois
1750,5.0,Tomislav Ivisic,C,24,23,25.1,4.0,8.0,0.497,1.7,...,4.0,5.7,1.6,0.3,0.6,1.4,1.9,10.5,NaN,Illinois
1751,6.0,Zvonimir Ivisic,F,27,4,17.8,2.5,4.9,0.519,1.0,...,3.8,5.1,0.3,0.3,2.3,0.7,1.4,6.9,NaN,Illinois
1752,7.0,Ben Humrichous,F,27,2,22.4,1.9,5.1,0.377,1.5,...,3.1,4.2,0.8,0.4,0.7,0.2,1.2,6.0,NaN,Illinois
1753,8.0,Jake Davis,F,27,12,19.8,1.8,4.2,0.434,1.6,...,1.2,2.1,0.8,0.3,0.2,0.2,1.3,5.7,NaN,Illinois
1754,9.0,Mihailo Petrovic,G,15,0,6.7,0.8,2.4,0.333,0.1,...,0.5,0.7,1.3,0.2,0.0,0.9,0.4,2.1,NaN,Illinois
1755,10.0,Brandon Lee,G,14,0,4.9,0.5,1.1,0.438,0.0,...,0.3,0.4,0.2,0.1,0.0,0.4,0.9,1.9,NaN,Illinois
